# Notebook 01 — Bronze Layer
**Source:** s3://brazilian-olist-s3-bucket/Dataset/ (read once, never again)
**Output:** 9 managed Delta tables in brazilian.bronze
**Rule:** Zero transformations. Raw data exactly as-is. Immutable audit trail.
After this notebook, S3 is never referenced again in this project.

In [0]:
from pyspark.sql import functions as F

spark.sql("USE CATALOG brazilian")
spark.sql("USE SCHEMA bronze")

SOURCE_PATH = "s3://brazilian-olist-s3-bucket/Dataset/"
TARGET_TABLE = "brazilian.bronze"

print(f"Source path   : {SOURCE_PATH}")
print(f"Target catalog: {TARGET_TABLE}")
print(f"Pattern       : {TARGET_TABLE}.<Dataset>")

In [0]:
# ── INGESTION FUNCTION ────────────────────────────────────────
#
# WHY A FUNCTION: All 9 CSVs follow the exact same pattern.
# Writing the logic once and calling it 9 times is cleaner,
# easier to debug, and professional — this is how data engineers
# write reusable pipeline code in production.
#
# WHAT IT DOES (line by line):
#   1. Builds the full S3 path by combining SOURCE_PATH + filename
#   2. Reads the CSV into a Spark DataFrame
#   3. Adds two metadata columns (_ingested_at, _source_file)
#      — standard production practice for data lineage
#   4. Writes the DataFrame as a managed Delta table
#      — saveAsTable() means Databricks owns the data files
#      — mode("overwrite") makes it safe to re-run
#   5. Counts and prints rows to confirm success

def ingest_to_bronze(filename, table_name, note=""):
    """
    Reads one CSV from S3.
    Writes it as a managed Delta table in brazilian.bronze.
    
    Args:
        filename   : exact CSV filename in S3 Dataset folder
        table_name : what to name the Delta table (no catalog/schema prefix)
        note       : optional data quality observation to print
    
    Returns:
        row count of the written table
    """

    # STEP 1: Build the full S3 file path
    # SOURCE_PATH already has trailing slash so we just concatenate
    path = f"{SOURCE_PATH}{filename}"

    # STEP 2: Read CSV into Spark DataFrame
    # header=True      → row 1 of CSV becomes column names
    # inferSchema=True → Spark detects int/string/float automatically
    # multiLine=True   → handles review text that contains line breaks
    # escape='"'       → handles commas inside quoted strings in CSV
    df = (spark.read
               .option("header",      "true")
               .option("inferSchema", "true")
               .option("multiLine",   "true")
               .option("escape",      '"')
               .csv(path))

    # STEP 3: Add metadata columns
    # WHY: In production pipelines, every Bronze table tracks WHEN
    # each row was ingested and WHERE it came from. This is called
    # data lineage — critical for debugging and auditing.
    # _ingested_at → timestamp of this specific pipeline run
    # _source_file → which CSV file this row came from
    df = (df
          .withColumn("_ingested_at", F.current_timestamp())
          .withColumn("_source_file",  F.lit(filename)))

    # STEP 4: Write as managed Delta table
    # format("delta")         → write as Delta Lake format (not plain CSV/Parquet)
    #                           Delta adds ACID transactions + time travel
    # mode("overwrite")       → replace table if it already exists
    #                           makes this cell safe to re-run
    # overwriteSchema=True    → update the schema if columns changed
    # saveAsTable(...)        → registers in Unity Catalog Hive Metastore
    #                           meaning: any notebook can now query this
    #                           table by name without knowing the file path
    #                           THIS is the difference between a file and a table
    full_table_name = f"{TARGET_TABLE}.{table_name}"

    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(full_table_name))

    # STEP 5: Count and confirm
    count = spark.table(full_table_name).count()
    cols  = len(df.columns) - 2  # subtract the 2 metadata cols we added

    print(f"  ✅  {full_table_name:<52} {count:>10,} rows  |  {cols} source cols")
    if note:
        print(f"        ⚠  {note}")

    return count

In [0]:
# ── INGEST ALL 9 TABLES ───────────────────────────────────────
#
# WHY THIS ORDER:
# olist_orders is the backbone — every other table joins to it
# We ingest it first so it's ready for spot-checking immediately
#
# Each call follows the pattern:
#   ingest_to_bronze(
#       "exact_filename_in_s3.csv",    ← must match S3 exactly
#       "table_name_in_databricks",    ← becomes brazilian.bronze.table_name
#       "data quality note"            ← printed below the row count
#   )

print("=" * 72)
print("BRONZE INGESTION")
print(f"Source : {SOURCE_PATH}")
print(f"Target : {TARGET_TABLE}.*")
print("=" * 72)

total_rows = 0

# TABLE 1 — Orders (backbone of the entire dataset)
# CRITICAL NOTE: customer_id in this table ≠ customer_unique_id
# in the customers table. Same person placing 3 orders gets
# 3 different customer_ids. This is the #1 data trap in Olist.
total_rows += ingest_to_bronze(
    "olist_orders_dataset.csv",
    "olist_orders",
    "customer_id here ≠ customer_unique_id — use unique_id for all analysis"
)

# TABLE 2 — Customers
# Contains BOTH customer_id and customer_unique_id
# customer_unique_id = the real person identifier — always use this
total_rows += ingest_to_bronze(
    "olist_customers_dataset.csv",
    "olist_customers",
    "customer_unique_id = real person. Same person gets new customer_id per order"
)

# TABLE 3 — Order Items
# Multiple rows per order_id — one row per product in the order
# Must aggregate to order level before joining to orders table
total_rows += ingest_to_bronze(
    "olist_order_items_dataset.csv",
    "olist_order_items",
    "Multiple rows per order_id — one row per product line item"
)

# TABLE 4 — Payments
# payment_value = actual money received — this is the Monetary in RFM
# One order can have multiple payment rows (installments / split payment)
total_rows += ingest_to_bronze(
    "olist_order_payments_dataset.csv",
    "olist_payments",
    "payment_value = actual money paid — Monetary source for RFM scoring"
)

# TABLE 5 — Reviews
# review_score 1-5 — satisfaction proxy and churn predictor
# Not all orders have reviews — expect ~75% coverage
# LEFT JOIN in Silver, do not drop orders with no review
total_rows += ingest_to_bronze(
    "olist_order_reviews_dataset.csv",
    "olist_reviews",
    "review_score 1-5 — satisfaction signal. Not all orders have reviews — LEFT JOIN"
)

# TABLE 6 — Products
# Category names are in Portuguese
# Will join with translation table in Silver for English labels
total_rows += ingest_to_bronze(
    "olist_products_dataset.csv",
    "olist_products",
    "Category names in Portuguese — join with olist_category_translation for English"
)

# TABLE 7 — Sellers
# Small table — ~3K sellers
# Used for geographic analysis in extended analysis notebook
total_rows += ingest_to_bronze(
    "olist_sellers_dataset.csv",
    "olist_sellers"
)

# TABLE 8 — Geolocation
# Largest file — ~1 million rows
# ZIP code to lat/lng mapping for Brazilian postal codes
# DO NOT CANCEL — let it run, takes 5-8 minutes
total_rows += ingest_to_bronze(
    "olist_geolocation_dataset.csv",
    "olist_geolocation",
    "~1M rows — largest file. Takes 5-8 mins. Do not cancel the cell."
)

# TABLE 9 — Category Translation
# Maps Portuguese product category names to English
# Tiny table — 71 rows
# Used in Silver when joining products for English category labels
total_rows += ingest_to_bronze(
    "product_category_name_translation.csv",
    "olist_category_translation",
    "Portuguese → English category mapping. 71 rows."
)

print()
print("=" * 72)
print(f"BRONZE COMPLETE")
print(f"Total rows ingested : {total_rows:,}")
print(f"Tables written to   : {TARGET_TABLE}.*")
print(f"S3 status           : done — will not be referenced again")
print("=" * 72)

In [0]:
# ── MASTER VERIFICATION ───────────────────────────────────────
#
# WHY: After writing all 9 tables, we confirm every single one
# exists and has the expected row count using the three-level
# namespace (catalog.schema.table) — the professional pattern
# in Unity Catalog that signals production-level thinking
#
# This query runs entirely inside Databricks — no S3 involved

print("BRONZE LAYER — FINAL VERIFICATION")
print("Using three-level namespace: catalog.schema.table\n")

spark.sql(f"""
    SELECT '{TARGET_TABLE}.olist_orders'               AS delta_table,
            COUNT(*)                                   AS rows
    FROM {TARGET_TABLE}.olist_orders

    UNION ALL SELECT '{TARGET_TABLE}.olist_customers',  COUNT(*)
    FROM {TARGET_TABLE}.olist_customers

    UNION ALL SELECT '{TARGET_TABLE}.olist_order_items', COUNT(*)
    FROM {TARGET_TABLE}.olist_order_items

    UNION ALL SELECT '{TARGET_TABLE}.olist_payments',    COUNT(*)
    FROM {TARGET_TABLE}.olist_payments

    UNION ALL SELECT '{TARGET_TABLE}.olist_reviews',     COUNT(*)
    FROM {TARGET_TABLE}.olist_reviews

    UNION ALL SELECT '{TARGET_TABLE}.olist_products',    COUNT(*)
    FROM {TARGET_TABLE}.olist_products

    UNION ALL SELECT '{TARGET_TABLE}.olist_sellers',     COUNT(*)
    FROM {TARGET_TABLE}.olist_sellers

    UNION ALL SELECT '{TARGET_TABLE}.olist_geolocation', COUNT(*)
    FROM {TARGET_TABLE}.olist_geolocation

    UNION ALL SELECT '{TARGET_TABLE}.olist_category_translation', COUNT(*)
    FROM {TARGET_TABLE}.olist_category_translation

    ORDER BY rows DESC
""").show(truncate=False)

print()
print("All 9 tables are now managed Delta tables inside Databricks.")
print("S3 path was used only to read raw CSVs. It is no longer needed.")
print("Notebooks 02 onwards read exclusively from brazilian.bronze.*")

In [0]:
# ── SPOT CHECK KEY COLUMNS ────────────────────────────────────
#
# WHY: Before moving to Silver, visually confirm that the most
# important columns look correct. This catches encoding issues,
# wrong column names, or bad date formats before they propagate.

print("SPOT CHECK 1 — olist_orders (notice both customer ID columns)")
spark.sql(f"""
    SELECT order_id,
           customer_id,
           order_status,
           order_purchase_timestamp
    FROM {TARGET_TABLE}.olist_orders
    LIMIT 4
""").show(truncate=False)

print("SPOT CHECK 2 — olist_customers (the critical ID distinction)")
spark.sql(f"""
    SELECT customer_id,
           customer_unique_id,
           customer_city,
           customer_state
    FROM {TARGET_TABLE}.olist_customers
    LIMIT 4
""").show(truncate=False)

print("SPOT CHECK 3 — olist_payments (payment_value is Monetary in RFM)")
spark.sql(f"""
    SELECT order_id,
           payment_type,
           payment_installments,
           payment_value
    FROM {TARGET_TABLE}.olist_payments
    LIMIT 4
""").show(truncate=False)

print("SPOT CHECK 4 — olist_reviews (review_score = churn signal)")
spark.sql(f"""
    SELECT order_id,
           review_score,
           review_comment_title
    FROM {TARGET_TABLE}.olist_reviews
    LIMIT 4
""").show(truncate=False)